In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║   CogVideoX-2B  |  INT8 Quantized  |  Free Colab T4         ║
# ║   FULLY FIXED VERSION — no black frames, no dtype errors     ║
# ║   Output: 720×480, 6 seconds @ 8 fps                         ║
# ╚══════════════════════════════════════════════════════════════╝


In [1]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — Install (then Runtime → Restart Session)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
!pip uninstall -y xformers torchao 2>/dev/null || true

!pip install -q \
    "torch==2.4.0" "torchvision==0.19.0" \
    --index-url https://download.pytorch.org/whl/cu121

!pip install -q \
    "diffusers==0.30.3" \
    "transformers==4.44.2" \
    "accelerate==0.33.0" \
    "imageio[ffmpeg]" \
    "sentencepiece"

print("✅ Done — NOW click Runtime → Restart Session")

✅ Done — NOW click Runtime → Restart Session


In [1]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Load pipeline (simple, no quantization)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch, gc, os
from diffusers import CogVideoXPipeline

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

print("\nLoading CogVideoX-2B pipeline...")

# float16 — trained in float16, safest choice on T4
pipe = CogVideoXPipeline.from_pretrained(
    "THUDM/CogVideoX-2b",
    torch_dtype=torch.float16,
)

# All 4 memory tricks together — proven to fit in T4
pipe.enable_model_cpu_offload()
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

gc.collect()
torch.cuda.empty_cache()

used = torch.cuda.memory_allocated()/1e9
total = torch.cuda.get_device_properties(0).total_memory/1e9
print(f"\n✅ Pipeline ready!  VRAM: {used:.1f}/{total:.1f} GB")

GPU  : Tesla T4
VRAM : 15.6 GB

Loading CogVideoX-2B pipeline...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/862M [00:00<?, ?B/s]

transformer/diffusion_pytorch_model.safe(…):   0%|          | 0.00/3.39G [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The config attributes {'invert_scale_latents': False} were passed to AutoencoderKLCogVideoX, but are not expected and will be ignored. Please verify your config.json configuration file.



✅ Pipeline ready!  VRAM: 0.1/15.6 GB


In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — Generate video  ← EDIT PROMPT HERE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch, gc

gc.collect()
torch.cuda.empty_cache()

PROMPT = """
A golden eagle soars over snow-capped mountains at golden hour.
Camera tracks it from behind, gliding through crisp alpine air.
Dramatic warm light, cinematic 4K nature documentary style.
"""

NEGATIVE_PROMPT = "blurry, low quality, watermark, text, distorted, static"

print("🎬 Generating... (~15 min on T4 at 50 steps)")

with torch.inference_mode():
    output = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        num_frames=49,
        num_inference_steps=25,
        guidance_scale=6.0,
        generator=torch.Generator(device="cuda").manual_seed(42),
    )

frames = output.frames[0]
print(f"✅ Done! {len(frames)} frames generated")

🎬 Generating... (~15 min on T4 at 50 steps)


  0%|          | 0/25 [00:00<?, ?it/s]

✅ Done! 49 frames generated


In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — Save + display
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import numpy as np, imageio
from IPython.display import Video, display

frames_np = []
for frame in frames:
    arr = np.array(frame, dtype=np.float32)
    if arr.max() <= 1.0:
        arr = arr * 255.0
    arr = np.nan_to_num(arr, nan=0.0, posinf=255.0, neginf=0.0)
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    frames_np.append(arr)

f0 = frames_np[0]
print(f"min:{f0.min()} max:{f0.max()} mean:{f0.mean():.1f}")

if f0.max() == 0:
    print("❌ Black frames — try seed=123 and rerun Cell 3")
else:
    imageio.mimsave("output.mp4", frames_np, fps=8,
                    codec="libx264", output_params=["-crf","18"])
    print("✅ Saved: output.mp4")
    display(Video("output.mp4", embed=True, width=720))

min:0 max:255 mean:207.7
✅ Saved: output.mp4
